# பாடம் 04 - கருவி பயன்பாடு வடிவமைப்பு மாதிரி

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework (Python) பயன்படுத்தி AI முகவர்கள் குறித்த **கருவி பயன்பாடு** வடிவமைப்பு மாதிரியை கற்றுக்கொள்ளப்போகிறீர்கள். நாம் கவனிக்கப்போகும் விஷயங்கள்:

- `@tool` அலங்காரக்கோவை மற்றும் வகைப்படுத்தப்பட்ட பராமரிப்பு உடன் செயல்பாட்டு கருவிகளை வரையறுத்தல்
- ஒவ்வொரு கருவியும் செய்யும் காரியங்களை மாதிரிக்கு தெரியப்படுத்த கருவி பொருளடக்கங்களை வழங்கல்
- `approval_mode` மூலம் கருவி செயல்பாட்டை கட்டுப்படுத்தல்
- Pydantic மாதிரிகள் மற்றும் `response_format` மூலம் **வழங்கப்பட்ட தகவலை அமைப்பிட்டு** திரும்ப வழங்குதல்

காட்சி அமைப்பு என்பது **பயண முன்பதிவு முகவர்** ஆகும், அது இடங்களை தேடலாம், கிடைக்கும் நிலையை சரிபார்க்கலாம், மற்றும் விமான தகவல்களை பெறலாம்.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool அலங்காரி உடன் கருவிகள் வரையறுத்தல்

`@tool` அலங்காரி ஒரு சாதாரண Python செயல்பாட்டை ஒரு முகவரால் அழைக்கக்கூடிய கருவியாக மாற்றுகிறது.  
முக்கிய புள்ளிகள்:

- **docstring** மாதிரி காணும் கருவி விளக்கமாக மாறுகிறது.
- **Type annotations** (விளக்கங்களுடன் `Annotated` உட்பட) கருவி வடிவமைப்பை வரையறுக்கின்றன.
- `approval_mode` செயல்படும் முன் ஒவ்வொரு அழைப்பையும் பயனர் ஒப்புதல் தேவைப்படுவதை கட்டுப்படுத்துகிறது.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## பல கருவிகளுடன் ஒரு முகவரியை உருவாக்குதல்

மாதிரி பயனர் கேள்விக்கு பதிலளிக்க தேவையான எந்த கருவியையும் அழைக்க அளவுருப்பெறிய வாடிக்கையாளருக்கு மூன்று கருவிகளையும் ஒப்படைக்கவும்.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## கருவிகளுடன் கட்டமைக்கப்பட்ட வெளியீடு

`response_format` ஐ Pydantic மாதிரியாக அமைப்பதன் மூலம், முகவர் சுயவிவர உரையை தவிர, நன்கு வகைப்படுத்தப்பட்ட JSON பொருளைப் பெற வலியுறுத்தப்படுகிறது. இது கீழ்தரக் குறியீடு முடிவுகளை நிரல் மூலம் உணர்வதற்குத் தேவையான போது பயனுள்ளதாகும்.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## கருவி ஒப்புதல் மாதிரிகள்

`@tool` இயங்குதளத்தின் `approval_mode` பராமரிப்பானது கருவி அழைப்புகள் நடைமுறைப்படுத்துவதற்கு முன் மனித ஒப்புதல் தேவைப்படுமா என்பதை கட்டுப்படுத்தும்:

| முறையும் | நடத்தை |
|---|---|
| `"never_require"` | கருவி தானாகவே இயங்கும் — பயனர் உறுதிப்படுத்தல் தேவையில்லை. |
| `"always_require"` | இயக்கத்திற்கு முன் ஒவ்வொரு அழைப்பும் பயனரால் ஒப்புதல் பெற வேண்டும். |

பக்கவிளைவுகள் உள்ள கருவிகள் (உதா: விமானம் தையல், கடன் அட்டை சார்ஜ்) க்கு `"always_require"` பயன்படுத்தவும், இதனால் மனிதர் செயல்முறையில் இருப்பர்.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் கற்றுக்கொண்டது:

1. **கருவிகள்** `@tool` டெக்கரேட்டர் பயன்படுத்தி Typed parameters மற்றும் கருவி ஸ்கீமாவாக சேவையாற்றும் டாக்ஸ்டிரிங்குகள் மூலம் வரையறுக்க.
2. **பல கருவிகளை சேர்த்து** முகவரி தொடர் முறையில் கூலையிட முடியும் என சிக்கலான கேள்விகளுக்கு பதில் அளிக்க.
3. **ஊட்டியமைக்கப்பட்ட வெளியீட்டினை** Pydantic மாடலை `response_format` ஆக காட்சியிட.
4. **கருவி அங்கீகாரம் கட்டுப்படுத்த** `approval_mode` பயன்படுத்தி உணர்ச்சி மிகுந்த செயல்பாடுகளுக்காக மனிதத்தை சுற்றிலும் வைக்க.

இவை வெளிப்புற அமைப்புகளுடன் பாதுகாப்பாக தொடர்பு கொள்ளக்கூடிய நம்பகமான, தயாரிப்பு-முனை முகவரிகளை உருவாக்க அடிப்படையாக செயல்படும்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
